
# Ambiente de ejecución

In [2]:
#Instalamos el paquete necesario para poder capturar gestionar tramas y paquetes de la red
!pip install scapy
#Importamos la biblioteca completa
from scapy.all import *

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 23.6 MB/s eta 0:00:00


In [3]:
#Obtenemos la dirección IP de la máquina propia y la MAC
my_ip = get_if_addr(conf.iface)
my_mac = get_if_hwaddr(conf.iface)
print("Local IP:", my_ip)
print("Local MAC:", my_mac)

Local IP: 172.28.0.12
Local MAC: 02:42:ac:1c:00:0c


In [4]:
#Recorremos la tabla de rutas para encontrar el IP del default gateway y su MAC
for route in conf.route.routes:
  if(route[0]==0 and route[3]=='eth0'):
    gw_ip = route[2]
    break
gw_mac = getmacbyip(gw_ip)
print("Gateway IP:", gw_ip)
print("Gateway MAC:", gw_mac)

Gateway IP: 172.28.0.1
Gateway MAC: 02:42:e1:87:0e:71


# Creamos una trama (frame)

In [5]:
# Seleccionamos un tipo de trama y la creamos
frame_type = "ICMP" # @param ["ARP","RARP","ICMP","UDP DNS", "TCP SYN", "TCP ACK", "VLAN"]
if frame_type=="ARP":
  # Armamos un paquete ARP (who has 192.168.1.1?)
  frame = Ether(src=my_mac, dst="ff:ff:ff:ff:ff:ff") / ARP(pdst=gw_ip)
elif frame_type=="RARP":
  # Armamos un paquete RARP (what is my IP address??)
  frame = Ether(src=my_mac, dst="ff:ff:ff:ff:ff:ff", type=0x8035) / ARP(op=3, pdst=gw_ip)
elif frame_type=="ICMP":
  # Armamos un paquete ICMP (ping)
  frame = Ether(src=my_mac, dst=gw_mac) / IP(dst=gw_ip) / ICMP()
elif frame_type=="UDP DNS":
  # Armamos un paquete UDP DNS Query
  frame = Ether(src=my_mac, dst=gw_mac) / IP(dst="8.8.8.8") / UDP(dport=53) / DNS(rd=1, qd=DNSQR(qname="www.example.com"))
elif frame_type=="TCP SYN":
  # Armamos un paquete TCP SYN mínimo
  frame = Ether(src=my_mac, dst=gw_mac) / IP(dst=gw_ip) / TCP(dport=80, flags="S")
elif frame_type=="TCP ACK":
  frame = Ether(src=my_mac, dst=gw_mac) / IP(dst=gw_ip) / TCP(dport=80, flags="A", options=[('Timestamp', (123,0))])
elif frame_type=="VLAN":
  # Armamos un paquete con VLAN
  frame = Ether(src=my_mac, dst=gw_mac) / Dot1Q(vlan=10) / IP(dst=gw_ip) / ICMP()

## Alterna: importamos una trama

In [ ]:
file_name = 'mi_paquete.pcapng'
try:
    imported_frames = rdpcap(file_name)
    print(f"Se han importado {len(imported_frames)} tramas del archivo {file_name}")
except FileNotFoundError:
    print(f"Error: El archivo {file_name} no fue encontrado.")
except Exception as e:
    print(f"Ocurrió un error al importar el archivo: {e}")
imported_frames
frame = imported_frames[0]

Se han importado 1 tramas del archivo mi_paquete.pcapng


## Analizamos la trama creada

In [ ]:
# Mostramos el resumen de la trama creada
print(len(frame), "bytes |", frame.summary())

100 bytes | Ether / IP / UDP / mDNS Qry b'_companion-link._tcp.local.'


In [ ]:
# Mostramos el detalle de la trama
frame.show()

###[ Ethernet ]###
  dst       = 01:00:5e:00:00:fb
  src       = f2:14:45:a0:89:dd
  type      = IPv4
###[ IP ]###
     version   = 4
     ihl       = 5
     tos       = 0x0
     len       = 86
     id        = 47409
     flags     = 
     frag      = 0
     ttl       = 255
     proto     = 17
     chksum    = 0xe15b
     src       = 172.18.147.251
     dst       = 224.0.0.251
     \options   \
###[ UDP ]###
        sport     = 5353
        dport     = 5353
        len       = 66
        chksum    = 0x8704
###[ DNS ]###
           id        = 0
           qr        = 0
           opcode    = QUERY
           aa        = 0
           tc        = 0
           rd        = 0
           ra        = 0
           z         = 0
           ad        = 0
           cd        = 0
           rcode     = ok
           qdcount   = 2
           ancount   = 0
           nscount   = 0
           arcount   = 0
           \qd        \
            |###[ DNS Question Record ]###
            |  qname     = 

In [ ]:
#También podemos analizar el frame en hexadecimal
print("Current frame length (bytes):",len(frame))
hexdump(frame)

Current frame length (bytes): 100
0000  01 00 5E 00 00 FB F2 14 45 A0 89 DD 08 00 45 00  ..^.....E.....E.
0010  00 56 B9 31 00 00 FF 11 E1 5B AC 12 93 FB E0 00  .V.1.....[......
0020  00 FB 14 E9 14 E9 00 42 87 04 00 00 00 00 00 02  .......B........
0030  00 00 00 00 00 00 0F 5F 63 6F 6D 70 61 6E 69 6F  ......._companio
0040  6E 2D 6C 69 6E 6B 04 5F 74 63 70 05 6C 6F 63 61  n-link._tcp.loca
0050  6C 00 00 0C 80 01 07 5F 72 64 6C 69 6E 6B C0 1C  l......_rdlink..
0060  00 0C 80 01                                      ....


## Agregamos relleno (padding) si es necesario


In [ ]:
def add_pad(frame, min_len=60, verbose=True):
    """
    Pads the frame with zeros if its length is less than min_len.
    Returns the (possibly padded) frame.
    """
    frame_bytes = bytes(frame)
    pad_len = max(0, min_len - len(frame_bytes))
    if pad_len > 0:
        if verbose:
            print(f"Frame size of {len(frame_bytes)} padded with {pad_len} bytes")
        frame = frame / Raw(b'\x00' * pad_len)
    else:
        if verbose:
            print("No padding required")
    return frame
base_frame = add_pad(frame)

No padding required


In [ ]:
#También podemos analizar el frame en hexadecimal
print("Complete frame (with padding if required):",len(base_frame),"bytes")
hexdump(base_frame)

Complete frame (with padding if required): 100 bytes
0000  01 00 5E 00 00 FB F2 14 45 A0 89 DD 08 00 45 00  ..^.....E.....E.
0010  00 56 B9 31 00 00 FF 11 E1 5B AC 12 93 FB E0 00  .V.1.....[......
0020  00 FB 14 E9 14 E9 00 42 87 04 00 00 00 00 00 02  .......B........
0030  00 00 00 00 00 00 0F 5F 63 6F 6D 70 61 6E 69 6F  ......._companio
0040  6E 2D 6C 69 6E 6B 04 5F 74 63 70 05 6C 6F 63 61  n-link._tcp.loca
0050  6C 00 00 0C 80 01 07 5F 72 64 6C 69 6E 6B C0 1C  l......_rdlink..
0060  00 0C 80 01                                      ....


## Agregamos el FCS a la trama

In [ ]:
import binascii

def add_fcs(frame, verbose=True):
    """
    Computes and appends the Ethernet FCS (CRC32, little-endian) to the frame.
    Returns the frame as bytes (with FCS appended).
    """
    frame_bytes = bytes(frame)
    # Compute CRC32 (Ethernet FCS, little-endian)
    crc = binascii.crc32(frame_bytes) & 0xFFFFFFFF
    fcs_bytes = crc.to_bytes(4, byteorder='little')
    if verbose:
        print(f"Appending FCS: {fcs_bytes.hex()} to frame of {len(frame_bytes)} bytes")
    return frame_bytes + fcs_bytes

# Agregamos el FCS a la trama
full_frame = add_fcs(base_frame)

Appending FCS: 780be583 to frame of 100 bytes


In [ ]:
#También podemos analizar el frame completo en hexadecimal
print("Full frame length (bytes):",len(full_frame))
hexdump(full_frame)

Full frame length (bytes): 104
0000  01 00 5E 00 00 FB F2 14 45 A0 89 DD 08 00 45 00  ..^.....E.....E.
0010  00 56 B9 31 00 00 FF 11 E1 5B AC 12 93 FB E0 00  .V.1.....[......
0020  00 FB 14 E9 14 E9 00 42 87 04 00 00 00 00 00 02  .......B........
0030  00 00 00 00 00 00 0F 5F 63 6F 6D 70 61 6E 69 6F  ......._companio
0040  6E 2D 6C 69 6E 6B 04 5F 74 63 70 05 6C 6F 63 61  n-link._tcp.loca
0050  6C 00 00 0C 80 01 07 5F 72 64 6C 69 6E 6B C0 1C  l......_rdlink..
0060  00 0C 80 01 78 0B E5 83                          ....x...


## Agregamos el Preambulo y SFD para crear un paquete

In [ ]:
def add_preamble_sfd(frame_bytes, verbose=True):
    """
    Prepends Ethernet preamble (7 bytes of 0x55) and SFD (1 byte of 0xD5) to frame bytes.
    Returns the resulting bytes object.
    """
    preamble = b'\x55' * 7
    sfd = b'\xD5'
    packet = preamble + sfd + frame_bytes
    if verbose:
        print(f"Preamble+SFD: {preamble.hex()} {sfd.hex()}")
        print(f"Ethernet packet (hex): {packet.hex()}")
    return packet, len(packet)
packet, packet_size = add_preamble_sfd(full_frame)

Preamble+SFD: 55555555555555 d5
Ethernet packet (hex): 55555555555555d501005e0000fbf21445a089dd080045000056b9310000ff11e15bac1293fbe00000fb14e914e9004287040000000000020000000000000f5f636f6d70616e696f6e2d6c696e6b045f746370056c6f63616c00000c8001075f72646c696e6bc01c000c8001780be583


In [ ]:
#También podemos analizar el paquete (y sus partes) en hexadecimal
print("Packet length (bytes):",len(packet))
hexdump(packet)

Packet length (bytes): 112
0000  55 55 55 55 55 55 55 D5 01 00 5E 00 00 FB F2 14  UUUUUUU...^.....
0010  45 A0 89 DD 08 00 45 00 00 56 B9 31 00 00 FF 11  E.....E..V.1....
0020  E1 5B AC 12 93 FB E0 00 00 FB 14 E9 14 E9 00 42  .[.............B
0030  87 04 00 00 00 00 00 02 00 00 00 00 00 00 0F 5F  ..............._
0040  63 6F 6D 70 61 6E 69 6F 6E 2D 6C 69 6E 6B 04 5F  companion-link._
0050  74 63 70 05 6C 6F 63 61 6C 00 00 0C 80 01 07 5F  tcp.local......_
0060  72 64 6C 69 6E 6B C0 1C 00 0C 80 01 78 0B E5 83  rdlink......x...


In [ ]:
def custom_hexdump(data, bytes_per_line=8):
    for offset in range(0, len(data), bytes_per_line):
        chunk = data[offset:offset+bytes_per_line]
        # Hex representation, grouped
        hex_str = ' '.join(f'{b:02x}' for b in chunk)
        # ASCII representation (printable chars or dot)
        ascii_str = ''.join(chr(b) if 32 <= b <= 126 else '.' for b in chunk)
        # Print with line number (offset)
        print(f"{offset:04x}: {hex_str:<23} {ascii_str}")

print("Packet length (bytes):",len(packet))
custom_hexdump(bytes(packet), bytes_per_line=8)

Packet length (bytes): 112
0000: 55 55 55 55 55 55 55 d5 UUUUUUU.
0008: 01 00 5e 00 00 fb f2 14 ..^.....
0010: 45 a0 89 dd 08 00 45 00 E.....E.
0018: 00 56 b9 31 00 00 ff 11 .V.1....
0020: e1 5b ac 12 93 fb e0 00 .[......
0028: 00 fb 14 e9 14 e9 00 42 .......B
0030: 87 04 00 00 00 00 00 02 ........
0038: 00 00 00 00 00 00 0f 5f ......._
0040: 63 6f 6d 70 61 6e 69 6f companio
0048: 6e 2d 6c 69 6e 6b 04 5f n-link._
0050: 74 63 70 05 6c 6f 63 61 tcp.loca
0058: 6c 00 00 0c 80 01 07 5f l......_
0060: 72 64 6c 69 6e 6b c0 1c rdlink..
0068: 00 0c 80 01 78 0b e5 83 ....x...


## Agregamos idles necesarios

In [ ]:
def add_idles_for_ipg(packet, min_ipg=12, idle_char=0x07, verbose=True):
    """
    Appends idle characters (idle_char) to 'packet' so that its length plus at least min_ipg idles is a multiple of 8 bytes.
    Returns the padded bytes object and number of idles inserted.
    """
    remainder = (len(packet) + min_ipg) % 8
    if remainder != 0:
        idles = 8 - remainder + min_ipg
    else:
        idles = min_ipg
    packet_padded = packet + bytes([idle_char] * idles)
    if verbose:
        print(f"Idles inserted: {idles}")
        print(f"Total length after idles: {len(packet_padded)} (multiple of 8: {len(packet_padded)%8 == 0})")
    return packet_padded, idles

packet_with_idles, idles = add_idles_for_ipg(packet)

Idles inserted: 16
Total length after idles: 128 (multiple of 8: True)


In [ ]:
print("Packet + idles length (bytes):",len(packet_with_idles))
custom_hexdump(bytes(packet_with_idles), bytes_per_line=8)

Packet + idles length (bytes): 128
0000: 55 55 55 55 55 55 55 d5 UUUUUUU.
0008: 01 00 5e 00 00 fb f2 14 ..^.....
0010: 45 a0 89 dd 08 00 45 00 E.....E.
0018: 00 56 b9 31 00 00 ff 11 .V.1....
0020: e1 5b ac 12 93 fb e0 00 .[......
0028: 00 fb 14 e9 14 e9 00 42 .......B
0030: 87 04 00 00 00 00 00 02 ........
0038: 00 00 00 00 00 00 0f 5f ......._
0040: 63 6f 6d 70 61 6e 69 6f companio
0048: 6e 2d 6c 69 6e 6b 04 5f n-link._
0050: 74 63 70 05 6c 6f 63 61 tcp.loca
0058: 6c 00 00 0c 80 01 07 5f l......_
0060: 72 64 6c 69 6e 6b c0 1c rdlink..
0068: 00 0c 80 01 78 0b e5 83 ....x...
0070: 07 07 07 07 07 07 07 07 ........
0078: 07 07 07 07 07 07 07 07 ........


## Reemplazamos por caracteres de control

In [ ]:
def add_delimiters(packet_bytes, idles, s_char=0xFB, t_char=0xFD, verbose=True):
    """
    Marks the packet start with /S/ and the first post-packet idle with /T/.

    - packet_bytes: bytes or bytearray of your full packet (preamble, SFD, payload, FCS, idles)
    - idles: number of trailing idle characters appended
    - s_char: byte value for /S/ (default 0xFB)
    - t_char: byte value for /T/ (default 0xFD)

    Returns:
        packet_array (bytearray): modified packet
        start_index (int): index of /S/
        term_index (int): index of /T/
    """
    packet_array = bytearray(packet_bytes)

    # 1. Replace first preamble byte with /S/
    packet_array[0] = s_char
    start_index = 0

    # 2. Replace the first idle after packet with /T/
    pkt_end = len(packet_array) - idles
    packet_array[pkt_end] = t_char
    term_index = pkt_end

    # 3. Optionally print info
    if verbose:
        print(f"/S/ inserted at index {start_index}")
        print(f"/T/ inserted at index {term_index}")

    return packet_array, start_index, term_index

packet_array, start_index, term_index = add_delimiters(packet_with_idles, idles)

/S/ inserted at index 0
/T/ inserted at index 112


In [ ]:
custom_hexdump(packet_array, bytes_per_line=8)

0000: fb 55 55 55 55 55 55 d5 .UUUUUU.
0008: 01 00 5e 00 00 fb f2 14 ..^.....
0010: 45 a0 89 dd 08 00 45 00 E.....E.
0018: 00 56 b9 31 00 00 ff 11 .V.1....
0020: e1 5b ac 12 93 fb e0 00 .[......
0028: 00 fb 14 e9 14 e9 00 42 .......B
0030: 87 04 00 00 00 00 00 02 ........
0038: 00 00 00 00 00 00 0f 5f ......._
0040: 63 6f 6d 70 61 6e 69 6f companio
0048: 6e 2d 6c 69 6e 6b 04 5f n-link._
0050: 74 63 70 05 6c 6f 63 61 tcp.loca
0058: 6c 00 00 0c 80 01 07 5f l......_
0060: 72 64 6c 69 6e 6b c0 1c rdlink..
0068: 00 0c 80 01 78 0b e5 83 ....x...
0070: fd 07 07 07 07 07 07 07 ........
0078: 07 07 07 07 07 07 07 07 ........


# Creamos la señal digital

In [ ]:
!pip install pyvcd

## Señal con sólo datos

In [ ]:
from vcd import VCDWriter
import datetime

def dump_data_vcd(
    packet_bytes,
    data_width=16,
    timescale='1ns',
    clk_period=10,
    filename="wide_data.vcd",
    scope="data_if"
):
    """
    Dumps packet bytes onto a wide data interface for VCD viewing.
    - packet_bytes: single bytes list or 'bytes' object.
    - data_width: bus width in bits (multiple of 8).
    The first byte goes to the LSB of each data word.
    """
    if data_width % 8 != 0:
        raise ValueError("data_width (bits) must be a multiple of 8")

    bytes_per_word = data_width // 8
    data = list(packet_bytes)
    # Split into words (little-endian within each word)
    words = [
        sum(data[i+j] << (8 * j) for j in range(bytes_per_word) if i+j < len(data))
        for i in range(0, len(data), bytes_per_word)
    ]

    with open(filename, 'w') as f:
        writer = VCDWriter(
            f,
            timescale=timescale,
            date=str(datetime.datetime.now()),
            comment="Wide data interface dump",
            version="1.0"
        )

        data_sig = writer.register_var(scope, 'data', 'wire', size=data_width)
        clk_sig = writer.register_var(scope, 'clk', 'wire', size=1)

        time = 0
        clk = 0
        writer.change(clk_sig, time, clk)  # initial state

        half_period = clk_period // 2
        for word in words:
            time += half_period
            clk = 1
            writer.change(clk_sig, time, clk)
            writer.change(data_sig, time, word)

            time += half_period
            clk = 0
            writer.change(clk_sig, time, clk)

        writer.close(time)
    print(f"VCD file '{filename}' created with wide {data_width}-bit data bus")


dump_data_vcd(
    packet_array,
    data_width=64,
    clk_period=5,
    filename="single_data_64b.vcd",
)

VCD file 'single_data_64b.vcd' created with wide 64-bit data bus


Descargamos el archivo VCD y lo visualizamos (https://app.surfer-project.org)

## Señal con datos y conrol (MII)

In [ ]:
def dump_mii_vcd(
    packet_bytes,
    packet_size,
    word_size_bytes,
    timescale='1ns',
    clk_period=10,
    filename="mii_wide_ctrl.vcd",
    scope="mii"
):
    """
    Creates VCD trace for wide MII/XGMII-style data+control interface.

    packet_bytes: full frame bytes (/S/, payload, /T/, idles)
    packet_size: number of meaningful bytes (excluding idles)
    word_size_bytes: width of the data bus in bytes
    """
    total = len(packet_bytes)
    assert packet_size < total, "packet_size must be less than total length"
    bus_width_bits = word_size_bytes * 8

    # Split into words
    words = [
        packet_bytes[i : i + word_size_bytes]
        for i in range(0, total, word_size_bytes)
    ]

    data_values = []
    ctrl_values = []

    for wi, word in enumerate(words):
        val = 0
        ctrl_mask = 0
        for bi, b in enumerate(word):
            g_idx = wi * word_size_bytes + bi
            val |= b << (8 * bi)
            # Determine if this is a control byte:
            # /S/ at g_idx == 0
            # /T/ at g_idx == packet_size
            # any subsequent idle g_idx > packet_size
            if g_idx == 0 or g_idx == packet_size or g_idx > packet_size:
                ctrl_mask |= (1 << bi)

        data_values.append(val)
        ctrl_values.append(ctrl_mask)

    with open(filename, 'w') as f:
        writer = VCDWriter(
            f, timescale=timescale,
            date=str(datetime.datetime.now()),
            comment="Wide MII w/ corrected control",
            version="1.0"
        )

        data_sig = writer.register_var(scope, 'data', 'wire', size=bus_width_bits)
        ctrl_sig = writer.register_var(scope, 'ctrl', 'wire', size=word_size_bytes)
        clk_sig = writer.register_var(scope, 'clk', 'wire', size=1)

        time = 0
        clk = 0
        writer.change(clk_sig, time, clk)
        half = clk_period // 2

        for d, c in zip(data_values, ctrl_values):
            # Rising edge: present data
            time += half
            clk = 1
            writer.change(clk_sig, time, clk)
            writer.change(data_sig, time, d)
            writer.change(ctrl_sig, time, c)

            # Falling edge
            time += half
            clk = 0
            writer.change(clk_sig, time, clk)

        writer.close(time)

    print(f"VCD MII generated: {filename}")


dump_mii_vcd(
    packet_bytes=packet_array,
    packet_size=packet_size,         # 1 /S/ + 58 payload + 1 /T/
    word_size_bytes=8,      # 64-bit interface
    filename="mii64.vcd",
)

VCD MII generated: mii64.vcd


Descargamos el archivo VCD y lo visualizamos (https://app.surfer-project.org)